In [0]:
from pyspark.sql import functions as F

In [0]:
CATALOG = "clinical_trial_supply_chain"

SILVER = f"{CATALOG}.silver"

SOURCE_TABLE = f"{SILVER}.fact_dispensing"
TARGET_TABLE = f"{SILVER}.fact_daily_consumption"

In [0]:
daily_consumption_df = (
    spark.table(SOURCE_TABLE)
    .groupBy(
        "dispense_date",
        "site_id",
        "study_id",
        "drug_type"
    )
    .agg(
        F.count("kit_id")
        .alias("total_kits_dispensed"),

        F.sum("quantity_dispensed")
        .alias("total_quantity_dispensed")
    )
)

In [0]:
daily_consumption_df = (
    daily_consumption_df
    .withColumnRenamed(
        "dispense_date",
        "consumption_date"
    )
)

In [0]:
(
    daily_consumption_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TARGET_TABLE)
)

In [0]:
print(
    f"fact_daily_consumption records : "
    f"{spark.table(TARGET_TABLE).count()}"
)

display(
    spark.table(TARGET_TABLE)
)